In [2]:
# basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
df_1 = pd.read_csv("scraped_steam_games.csv")
df_1.head()
#each csv is around 500 games long, will have to concatenate into one big dataframe


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,Counter-Strike 2,"Aug 21, 2012",Free To Play,No Discount,Valve,Valve,"9,767,778","8,401,860","1,365,918",Mostly Positive,"['Action', 'Free To Play']"
1,Palworld,"Jul 9, 2026",$29.99,No Discount,Pocketpair,Pocketpair,"466,604","440,779","25,825",Very Positive,"['Action', 'Adventure', 'Indie', 'RPG']"
2,Marvel Rivals,"Dec 5, 2024",Free To Play,No Discount,NetEase Games,NetEase Games,"405,517","307,825","97,692",Mixed,"['Action', 'Free To Play']"
3,Warframe,"Mar 25, 2013",Free To Play,No Discount,Digital Extremes,Digital Extremes,"678,937","593,693","85,244",Very Positive,"['Action', 'RPG', 'Free To Play']"
4,Baldur's Gate 3,"Aug 3, 2023",$59.99,$41.99,Larian Studios,Larian Studios,"850,871","823,561","27,310",Very Positive,"['Adventure', 'RPG', 'Strategy']"


In [4]:
df_2 = pd.read_csv("scraped_steam_games_2.csv")
df_2.head()

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,shapez 2 - Factory,"Apr 23, 2026",$29.99,$17.99,tobspr Games,tobspr Games,"14,985","14,546",439,Very Positive,"['Casual', 'Indie', 'Simulation', 'Strategy']"
1,Blue Archive,"Jul 3, 2025",Free To Play,No Discount,NEXON Games,NEXON Korea,"27,467","17,836","9,631",Overwhelmingly Negative,"['Adventure', 'Casual', 'RPG', 'Strategy']"
2,DRAGON BALL GEKISHIN SQUADRA,"Sep 9, 2025",Free To Play,No Discount,"GANBARION Co., Ltd.",Bandai Namco Entertainment,"10,606","6,904","3,702",Mixed,"['Action', 'Free To Play']"
3,Infection Free Zone,"Apr 11, 2024",$44.98,$17.31,Jutsu Games,Games Operators,"18,349","14,784","3,565",Very Positive,"['Action', 'Indie', 'Simulation', 'Strategy']"
4,Soul Land: Awakening World,"Jun 1, 2026",Free To Play,No Discount,37GAMES,37GAMES,348,167,181,Mixed,"['Action', 'Adventure', 'Massively Multiplayer..."


In [5]:
#concat the two dataframes together
df = pd.concat([df_1, df_2])
print(df.shape)

(597, 11)


# Data Cleaning

In [6]:
#go feature by feature
#date parse
df["Release Date"] = pd.to_datetime(df["Release Date"], format = "mixed")
print(df["Release Date"])

0     2012-08-21
1     2026-07-09
2     2024-12-05
3     2013-03-25
4     2023-08-03
         ...    
495   2026-01-27
496   2025-06-17
497   2025-09-08
498   2025-02-13
499   2021-04-08
Name: Release Date, Length: 597, dtype: datetime64[us]


In [7]:
#price
df["Price"] = df["Price"].replace("Free To Play", "0")
df["Price"] = df["Price"].replace(to_replace = r"(?i).*Free.*", value = "0", regex = True)
df["Price"] = df["Price"].replace("No Price", "0")
df["Price"] = df["Price"].str.replace("/ month", "", regex=False)
df["Price"] = df["Price"].replace(to_replace = r"(?i).*demo.*", value = "0", regex = True)
df["Price"] = df["Price"].str.strip("$")
df["Price"] = df["Price"].astype("float")


In [8]:
#discount
df["Discount"] = df["Discount"].replace("No Discount", "0")
df["Discount"] = df["Discount"].str.strip("$")
df["Discount"] = df["Discount"].astype("float")

In [9]:
df["Total Reviews"] = df["Total Reviews"].str.replace(",", "")
df["Total Reviews"] = df["Total Reviews"].astype("int64")

df["Positive Reviews"] = df["Positive Reviews"].str.replace(",", "")
df["Positive Reviews"] = df["Positive Reviews"].astype("int64")

df["Negative Reviews"] = df["Negative Reviews"].str.replace(",", "")
df["Negative Reviews"] = df["Negative Reviews"].astype("int64")

print(df.dtypes)


Name                           str
Release Date        datetime64[us]
Price                      float64
Discount                   float64
Developer                      str
Publisher                      str
Total Reviews                int64
Positive Reviews             int64
Negative Reviews             int64
Review Sentiment               str
Genre(s)                       str
dtype: object


In [10]:
#Separate the genres
import ast
df["Genre(s)"] = df["Genre(s)"].fillna("['None', 'None', 'None', 'None']")
df["Genre(s)"] = df["Genre(s)"].apply(ast.literal_eval)
df["Genre1"] = df["Genre(s)"].str[0]
df["Genre2"] = df["Genre(s)"].str[1]
df["Genre3"] = df["Genre(s)"].str[2]

df.head()


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9767778,8401860,1365918,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,466604,440779,25825,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie
2,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405517,307825,97692,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN
3,Warframe,2013-03-25,0.00,0.00,Digital Extremes,Digital Extremes,678937,593693,85244,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play
4,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,850871,823561,27310,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy


In [11]:
#Data Cleaning is complete

# New Feature Creation

In [12]:
#Year Released
df["Year Released"] = df["Release Date"].dt.year

In [14]:
#Discount Percentage
df["Discount %"] = round((df["Price"]-df["Discount"])/df["Price"] * 100)


In [15]:
df["Positive Reviews %"] = df["Positive Reviews"]/df["Total Reviews"] * 100
df["Negative Reviews %"] = df["Negative Reviews"]/df["Total Reviews"] * 100
df.head()

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3,Year Released,Discount %,Positive Reviews %,Negative Reviews %
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9767778,8401860,1365918,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN,2012,NaN,86.016083,13.983917
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,466604,440779,25825,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie,2026,100.0,94.465328,5.534672
2,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405517,307825,97692,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN,2024,NaN,75.909271,24.090729
3,Warframe,2013-03-25,0.00,0.00,Digital Extremes,Digital Extremes,678937,593693,85244,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play,2013,NaN,87.444490,12.555510
4,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,850871,823561,27310,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy,2023,30.0,96.790348,3.209652
